# Satu pembanding CUDA — input B yang sama

Buka notebook ini di **JupyterLab server NVIDIA**, dari paket `cuda_reference_*.zip`
yang diekspor di Mac. Di terminal server, ekstrak paket ke folder baru:

```bash
python3 -m zipfile -e cuda_reference_xxx.zip cuda-reference
```

Buka `cuda-reference/source/notebooks/08_cuda_reference.ipynb`, lalu Run All.
Default menjalankan instalasi runtime Python 3.11 terisolasi, menyediakan bobot
pinned, dan **satu** generasi B 30 detik: Base, seed 0, CFG 7, 50 langkah.
Baseline MPS dan audio B disalin utuh dari cache Mac. Tidak ada sweep, tambahan
metadata, perubahan sample rate input, atau inferensi panjang pada pembanding ini.

Server memerlukan ffmpeg/ffprobe dan ruang disk untuk runtime/bobot/cache;
siapkan sekitar 45–50 GiB untuk pemasangan baru. Jika bobot sudah ada pada lokasi
pinned, hash diverifikasi dan bobot dipakai ulang. CUDA 12.8 diperlukan oleh wheel
Linux x86_64 yang dipin; kecocokan driver/GPU diperiksa oleh runtime sebenarnya.
Notebook tidak menyewa GPU atau memasukkan kredensial GitHub: source snapshot
repo sudah ada dalam paket. Model upstream di-clone dari URL resmi pada revisi pinned.

**Batas kesimpulan:** dtype CUDA dipilih upstream sesuai GPU dan direkam; RNG CUDA
berbeda dari MPS meski seed 0 sama. Satu hasil CUDA bersih belum membuktikan bug MPS.


In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, uuid
from IPython.display import Audio, display
DRY_RUN = False  # True: verifikasi paket dan buat plan, tanpa instalasi/unduhan/inferensi
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
BUNDLE = ROOT.parent
assert (ROOT / "scripts/cuda_reference.py").is_file(), "Buka notebook dari source/notebooks pada paket ekspor"
manifest = json.loads((BUNDLE / "bundle.json").read_text())
assert manifest["schema_version"] == 1
for record in manifest["files"]:
    path = (BUNDLE / record["path"]).resolve()
    assert path.is_relative_to(BUNDLE.resolve()) and path.is_file(), "Path paket tidak valid"
    if "notebook_source_sha256" in record:
        book = json.loads(path.read_text())
        cells = [{"cell_type": c["cell_type"], "source": ("".join(c["source"]) if isinstance(c["source"], list) else c["source"]).replace("DRY_RUN = True", "DRY_RUN = False")} for c in book["cells"]]
        actual = hashlib.sha256(json.dumps(cells, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
        expected = record["notebook_source_sha256"]
    else:
        actual, expected = hashlib.sha256(path.read_bytes()).hexdigest(), record["sha256"]
    assert actual == expected, f"File paket berubah: {record['path']}"
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "tools/stem_eval"))
from cuda_reference import check_cuda, run_reference
from runtime_environment import runtime_environment
from setup_runtime import runtime_python, UPSTREAM
from stream_process import run_logged
from system_tools import require_audio_tools
require_audio_tools()
if not DRY_RUN:
    assert sys.platform.startswith("linux"), "Notebook referensi ini untuk server Linux CUDA"
    assert shutil.which("nvidia-smi"), "GPU NVIDIA tidak terlihat. Pilih server/kernel GPU sebelum instalasi."
    print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], text=True))
missing = sum(item["size"] for item in json.loads((ROOT / "configs/assets-base.json").read_text())["files"] if not (UPSTREAM / item["destination"]).is_file())
required = 2*missing + 5*1024**3 + (8*1024**3 if not runtime_python().is_file() else 0)
print("Commit snapshot:", manifest["bridge_commit"])
print("Input B SHA256:", manifest["source_sha256"])
print("Ruang kosong GiB:", round(shutil.disk_usage(ROOT).free/1024**3, 1), "| Estimasi ruang baru GiB:", round(required/1024**3, 1))
if not DRY_RUN:
    assert shutil.disk_usage(ROOT).free >= required, "Ruang disk belum cukup untuk runtime/cache dan bobot baru"
REFERENCE = BUNDLE / "reference"
WORK = ROOT / "runs" / ("cuda_reference_" + uuid.uuid4().hex[:12])
WORK.parent.mkdir(exist_ok=True)


## 1. Runtime pinned, terpisah dari kernel Jupyter

Jika runtime model sudah ada, gunakan dan verifikasi CUDA. Jika belum, setup
meng-clone upstream dan memasang dependency dari lockfile. PyTorch di kernel
notebook tidak dipakai untuk generasi. Tidak beralih diam-diam ke CPU.


In [ ]:
if DRY_RUN:
    print("DRY RUN: instalasi dan CUDA runtime check dilewati")
else:
    # uv sync is idempotent and also finishes runtimes left partially installed.
    run_logged([sys.executable, str(ROOT / "scripts/setup_runtime.py"), "--install"], env=runtime_environment(), check=True)
    print(json.dumps(check_cuda(runtime_python()), indent=2))


## 2. Bobot pretrained yang dipin

Paket transfer tidak memuat bobot. Pada server baru, aset harus diunduh atau
sudah disalin ke `source/vendor/ACE-Step-1.5/checkpoints/`. Unduhan dan verifikasi
menampilkan progress; file lengkap yang valid tidak diunduh ulang.


In [ ]:
if DRY_RUN:
    print("DRY RUN: tidak ada bobot diunduh atau model dimuat")
else:
    run_logged([str(runtime_python()), str(ROOT / "scripts/model_assets.py"),
                "--manifest", str(ROOT / "configs/assets-base.json"),
                "--upstream", str(UPSTREAM), "--download"], env=runtime_environment(), check=True)


## 3. Satu generasi CUDA dengan B dan konfigurasi baseline

Input tetap byte-identik, 32 kHz PCM16 yang sama dengan uji MPS. Ada resampling
internal upstream; native 48 kHz baru diuji setelah baseline ini, agar format
input tidak ikut menjadi variabel. BPM/key/time signature/chord/beat belum
menjadi input model pada pembanding ini. Report menyimpan perangkat, versi,
dtype aktual dan parameter yang benar-benar dipakai. Error/artefak bukan bukti
model berhasil hanya karena proses selesai.


In [ ]:
report = run_reference(REFERENCE, WORK, execute=not DRY_RUN)
print(json.dumps(report, indent=2))
print("Folder hasil:", WORK)


## 4. Bandingkan, lalu ekspor hasil

Dengarkan `mps_baseline_listen.wav` dan `cuda_listen.wav` pada perangkat dan
volume yang sama. Keduanya memakai gain 0.7. Jika CUDA bersih, perbedaan runtime/
dtype/RNG perlu ditelusuri; jika CUDA juga rusak, fokus pada model/input/
conditioning dan evaluasi kualitas. Sweep seed/CFG, metadata SongMap dan durasi
60–180 detik menunggu hasil satu pembanding ini.


In [ ]:
if report.get("status") == "generated":
    for name in ("mps_baseline_listen.wav", "cuda_listen.wav"):
        print(name)
        display(Audio(filename=str(WORK / name)))
    print("Decoder CUDA aktual:", json.dumps(report.get("decoder_capture", {}), indent=2))
else:
    print("Belum ada hasil inferensi CUDA nyata; status:", report.get("status"))


In [ ]:
import zipfile
result_zip = WORK / "cuda_result.zip"
paths = [WORK / name for name in ("cuda_reference.json", "session.json", "mps_baseline_listen.wav", "cuda_listen.wav")]
paths += list((WORK / "capture").glob("*.json")) + list((WORK / "capture").glob("*.npy"))
paths += list((WORK / "generation").glob("*/result.json"))
with zipfile.ZipFile(result_zip, "x", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in paths:
        if path.is_file():
            archive.write(path, arcname=str(path.relative_to(WORK)))
print("Paket hasil tanpa bobot model:", result_zip)
